[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/en_clase/S09_02_pca_en_clase.ipynb)

# S09 · 02 — Reducir dimensiones con PCA · versión en clase

**Tiempo estimado:** 30 min en clase · **Sprint 9** · M3 (UD 3.10, PCA con mención a t-SNE)

📖 *¿El código se hace largo? Léelo [explicado por escrito](https://github.com/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/codigo_explicado/S09_02_pca.md): recorrido, glosario y qué hace cada celda.* <!--codigo-explicado-->

**Contexto TurisData.** Dirección quiere *ver* a sus huéspedes en una sola imagen, pero
tenemos 7 variables por persona y no se puede dibujar en 7 dimensiones. **PCA** (análisis de
componentes principales) resume muchas variables en pocas "variables nuevas" (componentes) que
conservan la mayor parte de la información. Sirve para **visualizar**, para **quitar ruido** y
para **acelerar** otros modelos.

**Al terminar sabrás:**
1. Qué hace PCA con una analogía sencilla y cuándo usarlo.
2. Leer la varianza explicada y decidir cuántos componentes conservar.
3. Dibujar los huéspedes en 2D y colorearlos por segmento.
4. Interpretar las "cargas": qué variables componen cada componente.
5. Saber qué es t-SNE y cuándo *no* fiarse de sus gráficos.

> 🏫 **Versión «en clase»** · ⏱ ≈ 30 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **1, 2**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_02_pca.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

huespedes = pd.read_csv(dato("huespedes.csv"))
variables = [c for c in huespedes.columns if c != "id_huesped"]
X = StandardScaler().fit_transform(huespedes[variables])
grupo = KMeans(n_clusters=4, n_init=10, random_state=SEMILLA).fit_predict(X)
print(X.shape)

## 1. La idea de PCA

**Analogía.** Imagina una nube de puntos con forma de puro. Para describirla no necesitas los tres
ejes de la habitación: te basta **un eje a lo largo del puro** (donde hay más variación) y otro
más corto a lo ancho. PCA encuentra esos nuevos ejes:
- el **componente 1** es la dirección con **más variación** de los datos,
- el **componente 2** es la siguiente, **perpendicular** a la primera, y así sucesivamente.

Cada componente es una **mezcla** de las variables originales, y viene con un porcentaje de
**varianza explicada** (cuánta información conserva). **Siempre se escalan antes** las variables.

**Ejemplo resuelto:** PCA con todos los componentes y su varianza explicada.

In [ ]:
pca_completo = PCA(random_state=SEMILLA).fit(X)
varianza = pca_completo.explained_variance_ratio_
for i, v in enumerate(varianza, start=1):
    print(f"Componente {i}: {v:.1%}")

### Ejercicio 1 · 🏫 en clase
Calcula en `varianza_acum` la varianza explicada **acumulada** (usa `np.cumsum` sobre `varianza`)
y en `n_comp_90` el **número mínimo de componentes** que explican al menos el 90 % de la varianza.

In [ ]:
# TODO: np.cumsum da la acumulada; para el mínimo de componentes, busca la primera posición que llega a 0,90 y recuerda que se cuenta desde 1
...

assert abs(varianza_acum[-1] - 1) < 1e-9 and len(varianza_acum) == 7, "varianza_acum debe acabar en 1 y tener 7 valores"
assert n_comp_90 == 5, "Revisa: busca el primer componente donde la acumulada llega a 0.90 y cuéntalo desde 1"
print("Ejercicio 1 correcto ✔ — con", n_comp_90, "componentes conservamos el 90 % de la información")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(range(1, 8), varianza)
ax[0].set_title("Varianza explicada por componente")
ax[0].set_xlabel("Componente"); ax[0].set_ylabel("Proporción")
ax[1].plot(range(1, 8), varianza_acum, marker="o")
ax[1].axhline(0.9, color="tab:red", linestyle="--", label="90 %")
ax[1].set_title("Varianza explicada acumulada")
ax[1].set_xlabel("Número de componentes"); ax[1].legend()
plt.tight_layout(); plt.show()

## 2. Ver a los huéspedes en 2D

**Ejemplo resuelto:** proyectamos a 2 componentes.

In [ ]:
pca2 = PCA(n_components=2, random_state=SEMILLA)
Z2 = pca2.fit_transform(X)
print("Forma:", Z2.shape, "· información conservada:", f"{pca2.explained_variance_ratio_.sum():.1%}")

### Ejercicio 2 · 🏫 en clase
Dibuja un diagrama de dispersión de `Z2` (columna 0 en el eje X, columna 1 en el eje Y) coloreando
cada punto según la variable `grupo` (el segmento de k-means). Guarda la figura en `fig2d`.
Pon título y nombres de ejes con el porcentaje de varianza de cada componente.

In [ ]:
# TODO: Crea la figura con plt.subplots y usa ax.scatter con dos columnas de Z2; el argumento c sirve para colorear por grupo
...

assert len(fig2d.axes[0].collections) >= 1, "fig2d debe contener un scatter"
assert fig2d.axes[0].get_title() != "", "Añade un título al gráfico"
plt.show()

Los segmentos se ven **separados** en el plano: PCA nos permite *comprobar a ojo* que el clustering
tiene sentido. Ojo: 2 componentes conservan un ~69 % de la información; lo que ves es una
**sombra** de los datos, no toda la verdad.

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_02_pca.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
